In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from transformers import Trainer
from transformers import DataCollatorWithPadding
import json
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [ ]:
with open("final_triplets_dataset.json", "r") as f:
    records = json.load(f)

rows = []
for r in records:
    t = r["triplets"]
    rows.append({
        "evidence_id":           r["id"],
        "supported":    t["supported"],
        "refuted":      t["refuted"]["statement"],
        "unverifiable": t["unverifiable"]["statement"]
    })

gpt4o_df = pd.DataFrame(rows)

evidence = pd.read_csv('filtered_climate_evidence - filtered_climate_evidence.csv')
gpt4o_df = pd.merge(gpt4o_df, evidence[['evidence_id', 'evidence_text']], how='left',
                    on='evidence_id')
human_df = pd.read_csv('Human - test.csv')

In [ ]:
# Combine human and gpt4o data
df = pd.concat([gpt4o_df, human_df], ignore_index=True)

# Split by evidence_id BEFORE expanding
unique_evidence_ids = df['evidence_id'].unique()
train_ids, test_ids = train_test_split(
    unique_evidence_ids,
    test_size=0.1,
    random_state=42
)

train_base = df[df['evidence_id'].isin(train_ids)]
test_base = df[df['evidence_id'].isin(test_ids)]

In [ ]:
def expand_rows(df):

    rows = []

    for _, r in df.iterrows():

        rows.append({
            "context": r["evidence_text"],
            "statement": r["supported"],
            "label": 0
        })

        rows.append({
            "context": r["evidence_text"],
            "statement": r["refuted"],
            "label": 1
        })

        rows.append({
            "context": r["evidence_text"],
            "statement": r["unverifiable"],
            "label": 2
        })

    return pd.DataFrame(rows)


train_df = expand_rows(train_base)
test_df = expand_rows(test_base)

train_df = train_df.dropna(subset=["context", "statement", "label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["context", "statement", "label"]).reset_index(drop=True)

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("Train label distribution:\n", train_df["label"].value_counts())
print(f"Train examples: {len(train_df)}")
print("\nTest label distribution:\n", test_df["label"].value_counts())
print(f"Test examples: {len(test_df)}")

Train label distribution:
 label
0    5054
1    5054
2    5054
Name: count, dtype: int64
Train examples: 15162

Test label distribution:
 label
0    564
1    564
2    564
Name: count, dtype: int64
Test examples: 1692


In [ ]:
# Shuffle to prevent ordering effects
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

# Use RoBERTa instead - better for subtle differences
model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(example):
    return tokenizer(
        example["context"],
        example["statement"],
        truncation=True,
        max_length=512,
        padding=False
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
train_dataset = train_dataset.remove_columns(["context", "statement"])
test_dataset = test_dataset.remove_columns(["context", "statement"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/15162 [00:00<?, ? examples/s]

Map:   0%|          | 0/1692 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-6,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_dir="./logs",
    logging_steps=100,
    warmup_steps=100,
    weight_decay=0.01,
    max_grad_norm=1.0,
    metric_for_best_model="f1"
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    cm = confusion_matrix(labels, preds)

    print("Confusion Matrix:")
    print(cm)

    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="macro")
    }

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator
)

trainer.train()

trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.278117,0.132195,0.966903,0.966902
2,0.237824,0.160373,0.971631,0.971633
3,0.222688,0.161918,0.973404,0.973391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Confusion Matrix:
[[544  20   0]
 [ 25 534   5]
 [  3   3 558]]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Confusion Matrix:
[[552  12   0]
 [ 24 536   4]
 [  2   3 559]]


Confusion Matrix:
[[552  12   0]
 [ 24 536   4]
 [  2   3 559]]


{'eval_loss': 0.16191837191581726,
 'eval_accuracy': 0.973404255319149,
 'eval_f1': 0.9733914061148178,
 'eval_runtime': 33.2599,
 'eval_samples_per_second': 50.872,
 'eval_steps_per_second': 6.374,
 'epoch': 3.0}